# EmoSet Axis Calibration Notebook (Gaussian Bayes, CLIP+DINO)

This notebook does exactly this:
1. Loads EmoSet + cached CLIP/DINO embeddings.
2. Defines one axis per emotion from fixed CLIP pos/neg prompts.
3. Builds prior axes and computes the axis-vs-ground-truth correlation matrix.
4. Samples **3 fixed random images per emotion**.
5. Applies moves per axis with target: **100% if GT == axis else 0%**.
6. Recomputes and visualizes correlation matrices after **every move**.

Use the **Hyperparameters** cell to tune Bayes behavior quickly.


### Imports
This cell imports the Python modules and configures the backend path so the notebook can access project code.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Make backend imports work whether notebook is run from repo root or backend/
CWD = Path.cwd().resolve()
if (CWD / 'backend').exists():
    REPO_ROOT = CWD
elif CWD.name == 'backend' and (CWD.parent / 'backend').exists():
    REPO_ROOT = CWD.parent
else:
    REPO_ROOT = CWD

sys.path.insert(0, str(REPO_ROOT / 'backend'))

from gallery_backend import ImageGalleryEngine


### Hyperparameters
This cell defines the dataset path, text encoder settings, Bayesian hyperparameters, and fixed prompt templates used throughout the notebook.


In [ ]:
# =========================
# Hyperparameters (tunable)
# =========================

DATASET_ROOT = REPO_ROOT / 'data' / 'datasets' / 'EmoSet'

# Set CLIP_MODEL_NAME to a local path for offline runs.
# Example: CLIP_MODEL_NAME = '/path/to/local/clip-vit-base-patch32'
CLIP_MODEL_NAME = 'openai/clip-vit-base-patch32'
CLIP_LOCAL_FILES_ONLY = False

# Axis Bayes settings
AXIS_BAYES_MODE = 'gaussian'  # 'gaussian' | 'rank' | 'graph'
AXIS_BAYES_FEATURE_SPACE = 'clip_dino'  # 'clip' | 'clip_dino'
AXIS_BAYES_CLIP_WEIGHT = 0.70
AXIS_BAYES_DINO_WEIGHT = 0.30
AXIS_BAYES_ALPHA = 96.0
AXIS_BAYES_DINO_ALPHA = 220.0
AXIS_BAYES_BIAS_ALPHA = 1.0
AXIS_BAYES_SIGMA2 = 0.04
AXIS_BAYES_GRAPH_KNN_K = 16
AXIS_BAYES_GRAPH_LAMBDA_SMOOTH = 6.0
AXIS_BAYES_GRAPH_LAMBDA_PRIOR = 1.0
AXIS_BAYES_GRAPH_JITTER = 1e-6
AXIS_BAYES_RANK_ETA = 0.25
AXIS_BAYES_RANK_ANCHOR_K = 6
AXIS_BAYES_RANK_ANCHOR_DELTA = 0.12
AXIS_BAYES_RANK_MAX_PAIRS = 100
AXIS_BAYES_MAX_MOVES = 0
AXIS_BAYES_HOTSPOT_BOUNDARY = 50.0
AXIS_BAYES_HOTSPOT_TAU = 18.0
AXIS_BAYES_HOTSPOT_K = 10
AXIS_BAYES_EXEMPLAR_K = 4

# Move protocol settings
MOVES_PER_EMOTION = 3
MOVE_SELECTION_SEED = 123

# Fixed prompt endpoints (edit as needed)
NEG_PROMPT_COMMON = 'an emotionally neutral image with no clear emotion'
POS_PROMPT_TEMPLATE = 'an image that strongly conveys {emotion}'

# Optional explicit prompts per emotion. Leave empty dict to use templates.
POS_PROMPTS_OVERRIDE = {
    # 'anger': 'an image that strongly conveys anger',
}
NEG_PROMPTS_OVERRIDE = {
    # 'anger': 'an emotionally neutral image with no anger',
}


### Core Utilities
This cell defines the reusable normalization, quantile mapping, Gaussian update, and correlation plotting helpers.


In [ ]:
def l2_normalize_rows(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)
    n = np.linalg.norm(x, axis=1, keepdims=True) + 1e-8
    return (x / n).astype(np.float32)


def l2_normalize_vec(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32).reshape(-1)
    n = float(np.linalg.norm(x))
    if n <= 1e-8:
        raise ValueError('Cannot normalize zero vector')
    return (x / n).astype(np.float32)


def quantile_from_sorted(z_sorted: np.ndarray, p01: float) -> float:
    z = np.asarray(z_sorted, dtype=np.float32).reshape(-1)
    n = z.size
    if n == 0:
        return 0.0
    if n == 1:
        return float(z[0])
    p = float(np.clip(float(p01), 0.0, 1.0))
    pos = p * (n - 1)
    lo = int(np.floor(pos))
    hi = int(np.ceil(pos))
    if lo == hi:
        return float(z[lo])
    t = float(pos - lo)
    return float((1.0 - t) * z[lo] + t * z[hi])


def pearson_corr(x: np.ndarray, y: np.ndarray) -> float:
    x = np.asarray(x, dtype=np.float64).reshape(-1)
    y = np.asarray(y, dtype=np.float64).reshape(-1)
    if x.size != y.size or x.size == 0:
        return np.nan
    x = x - x.mean()
    y = y - y.mean()
    den = np.linalg.norm(x) * np.linalg.norm(y)
    if den <= 1e-12:
        return 0.0
    return float(np.dot(x, y) / den)


def gaussian_axis_update(
    X_all: np.ndarray,
    w0: np.ndarray,
    z0_sorted: np.ndarray,
    moves: list,
    alpha_vec: np.ndarray,
    bias_alpha: float,
    sigma2: float,
    b0: float = 0.0,
):
    """
    moves: list[(image_index, target_percentile_01)]
    """
    w0 = np.asarray(w0, dtype=np.float32)
    alpha_vec = np.asarray(alpha_vec, dtype=np.float32)
    alpha_inv = 1.0 / np.maximum(alpha_vec, 1e-8)

    if len(moves) == 0:
        return w0.copy(), float(b0)

    idx = np.asarray([int(i) for i, _ in moves], dtype=np.int64)
    p01 = [float(p) for _, p in moves]

    X = X_all[idx, :].astype(np.float32)
    y_raw = np.asarray([quantile_from_sorted(z0_sorted, p) for p in p01], dtype=np.float32)

    m = X.shape[0]
    ones = np.ones((m,), dtype=np.float32)
    XS = (X * alpha_inv[None, :]).astype(np.float32)
    A = (
        (float(sigma2) * np.eye(m, dtype=np.float32))
        + (XS @ X.T)
        + ((1.0 / float(bias_alpha)) * np.outer(ones, ones).astype(np.float32))
    )
    A_inv = np.linalg.inv(A + (1e-6 * np.eye(m, dtype=np.float32)))
    r = y_raw - (X @ w0) - float(b0)

    mu = w0 + (alpha_inv * (X.T @ (A_inv @ r)))
    b = float(b0) + float((1.0 / float(bias_alpha)) * (ones @ (A_inv @ r)))

    if float(np.dot(mu, w0)) < 0.0:
        mu = -mu
        b = -b

    n = float(np.linalg.norm(mu))
    if n > 1e-8:
        mu = (mu / n).astype(np.float32)
        b = float(b / n)
    else:
        mu = w0.copy()
        b = float(b0)
    return mu, b


def plot_corr_matrix(corr: np.ndarray, row_labels: list, col_labels: list, title: str):
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(corr, cmap='RdBu_r', vmin=-1.0, vmax=1.0)
    ax.set_xticks(np.arange(len(col_labels)))
    ax.set_yticks(np.arange(len(row_labels)))
    ax.set_xticklabels(col_labels, rotation=45, ha='right')
    ax.set_yticklabels(row_labels)
    ax.set_title(title)
    for i in range(corr.shape[0]):
        for j in range(corr.shape[1]):
            ax.text(j, i, f'{corr[i, j]:.2f}', ha='center', va='center', fontsize=8)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()


### EmoSet Labels
This cell loads the EmoSet image list and ground-truth emotion labels from metadata or filenames.


In [ ]:
# -------------------------
# Load EmoSet + GT labels
# -------------------------

dataset_root = Path(DATASET_ROOT)
if not dataset_root.exists():
    raise FileNotFoundError(f'Dataset root not found: {dataset_root}')

engine = ImageGalleryEngine(str(dataset_root))
entries = engine.list_images()
ids = [str(e.id) for e in entries]
id_to_idx = {img_id: i for i, img_id in enumerate(ids)}

metadata_path = dataset_root / 'metadata.csv'
if metadata_path.exists():
    meta = pd.read_csv(metadata_path)
    meta.columns = [str(c).strip() for c in meta.columns]
    image_col = 'image' if 'image' in meta.columns else meta.columns[0]
    emotion_col = 'emotion' if 'emotion' in meta.columns else None
    if emotion_col is None:
        for c in meta.columns:
            if c.lower().strip() == 'emotion':
                emotion_col = c
                break
    if emotion_col is None:
        raise RuntimeError(f'Could not find emotion column in metadata: {meta.columns.tolist()}')
    meta[image_col] = meta[image_col].astype(str).str.strip()
    meta[emotion_col] = meta[emotion_col].astype(str).str.strip().str.lower()
    id_to_emotion = dict(zip(meta[image_col], meta[emotion_col]))
else:
    # Fallback from filename prefix if metadata is absent
    id_to_emotion = {img_id: img_id.split('_', 1)[0].strip().lower() for img_id in ids}

labels = []
missing = []
for img_id in ids:
    emo = id_to_emotion.get(img_id)
    if emo is None or str(emo).strip() == '':
        missing.append(img_id)
        emo = img_id.split('_', 1)[0].strip().lower()
    labels.append(emo)

if missing:
    print(f'Warning: missing labels for {len(missing)} images; inferred from filename prefix.')

labels = np.asarray(labels)
emotions = sorted(pd.unique(labels).tolist())
print(f'n_images={len(ids)}')
print(f'emotions={emotions}')


### Embedding Loading
This cell loads cached CLIP and DINO embeddings and builds the fused feature matrix used by the axis methods.


In [ ]:
# -------------------------
# Load CLIP (+ optional DINO) embeddings and fuse
# -------------------------

clip_emb = engine._load_embeddings_only(entries, method='clip')
if clip_emb is None:
    raise RuntimeError('CLIP cache not found. Expected .cache/embeddings_clip.npz')
X_clip = l2_normalize_rows(np.asarray(clip_emb, dtype=np.float32))

if AXIS_BAYES_FEATURE_SPACE == 'clip_dino':
    dino_emb = engine._load_embeddings_only(entries, method='dino')
    if dino_emb is None:
        raise RuntimeError('DINO cache not found. Expected .cache/embeddings_dino.npz')
    X_dino = l2_normalize_rows(np.asarray(dino_emb, dtype=np.float32))
    if X_dino.shape[0] != X_clip.shape[0]:
        raise RuntimeError(f'CLIP/DINO count mismatch: {X_clip.shape[0]} vs {X_dino.shape[0]}')

    cw = float(max(1e-6, AXIS_BAYES_CLIP_WEIGHT))
    dw = float(max(1e-6, AXIS_BAYES_DINO_WEIGHT))
    ws = cw + dw
    clip_scale = float(np.sqrt(cw / ws))
    dino_scale = float(np.sqrt(dw / ws))

    X = np.concatenate([clip_scale * X_clip, dino_scale * X_dino], axis=1).astype(np.float32)
    clip_dim = X_clip.shape[1]
    dino_dim = X_dino.shape[1]
    alpha_vec = np.concatenate([
        np.full((clip_dim,), float(AXIS_BAYES_ALPHA), dtype=np.float32),
        np.full((dino_dim,), float(AXIS_BAYES_DINO_ALPHA), dtype=np.float32),
    ], axis=0)
else:
    X = X_clip.astype(np.float32)
    clip_dim = X_clip.shape[1]
    dino_dim = 0
    clip_scale = 1.0
    dino_scale = 0.0
    alpha_vec = np.full((clip_dim,), float(AXIS_BAYES_ALPHA), dtype=np.float32)

print(f'X shape={X.shape} (clip_dim={clip_dim}, dino_dim={dino_dim})')
print(f'clip_scale={clip_scale:.4f}, dino_scale={dino_scale:.4f}')


### Text Priors
This cell encodes positive and negative emotion prompts and builds one prior semantic axis per emotion.


In [ ]:
# -------------------------
# Build prior axis per emotion from fixed CLIP texts
# -------------------------
import torch
from transformers import CLIPModel, CLIPTokenizer


def build_clip_text_encoder(model_name: str, local_only: bool):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = CLIPModel.from_pretrained(model_name, use_safetensors=True, local_files_only=local_only).to(device)
    tokenizer = CLIPTokenizer.from_pretrained(model_name, local_files_only=local_only)
    model.eval()

    def encode(text: str) -> np.ndarray:
        with torch.no_grad():
            tokens = tokenizer([text], return_tensors='pt', padding=True, truncation=True)
            tokens = {k: v.to(device) for k, v in tokens.items()}
            feats = model.get_text_features(**tokens)
            feats = feats / (feats.norm(dim=-1, keepdim=True) + 1e-8)
        return feats.squeeze(0).detach().cpu().numpy().astype(np.float32)

    return encode


encode_text = build_clip_text_encoder(CLIP_MODEL_NAME, CLIP_LOCAL_FILES_ONLY)

pos_prompts = {
    emo: POS_PROMPTS_OVERRIDE.get(emo, POS_PROMPT_TEMPLATE.format(emotion=emo))
    for emo in emotions
}
neg_prompts = {
    emo: NEG_PROMPTS_OVERRIDE.get(emo, NEG_PROMPT_COMMON)
    for emo in emotions
}

prior = {}  # emotion -> dict(w0, z0, z0_sorted)
for emo in emotions:
    pos = l2_normalize_vec(encode_text(pos_prompts[emo]))
    neg = l2_normalize_vec(encode_text(neg_prompts[emo]))
    w_clip = l2_normalize_vec(pos - neg)

    if AXIS_BAYES_FEATURE_SPACE == 'clip_dino':
        w0 = l2_normalize_vec(np.concatenate([w_clip, np.zeros((dino_dim,), dtype=np.float32)], axis=0))
    else:
        w0 = w_clip

    z0 = (X @ w0).astype(np.float32)
    prior[emo] = {
        'w0': w0.astype(np.float32),
        'b0': 0.0,
        'z0': z0,
        'z0_sorted': np.sort(z0).astype(np.float32),
    }

pd.DataFrame({'emotion': emotions, 'pos_prompt': [pos_prompts[e] for e in emotions], 'neg_prompt': [neg_prompts[e] for e in emotions]})


### Prior Correlations
This cell computes and visualizes the correlation matrix between prior axis scores and ground-truth emotions.


In [ ]:
# -------------------------
# Correlation matrix helper: rows=axes, cols=GT emotions
# -------------------------

Y = np.stack([(labels == emo).astype(np.float32) for emo in emotions], axis=1)  # n x n_emotions


def compute_corr_matrix(axis_state_dict):
    # axis_state_dict[axis_emotion] = {'mu':..., 'b':...}
    rows = []
    for axis_emo in emotions:
        st = axis_state_dict[axis_emo]
        z = (X @ st['mu']) + float(st['b'])
        row = [pearson_corr(z, Y[:, j]) for j in range(Y.shape[1])]
        rows.append(row)
    return np.asarray(rows, dtype=np.float32)


prior_states = {emo: {'mu': prior[emo]['w0'].copy(), 'b': 0.0} for emo in emotions}
C_prior = compute_corr_matrix(prior_states)
plot_corr_matrix(C_prior, emotions, emotions, 'Prior correlation matrix (axis projection vs GT one-vs-rest)')


### Move Catalog
This cell samples the fixed set of GT-based image moves that will be replayed for every axis during refinement.


In [ ]:
# -------------------------
# Fixed move set: 3 random images per emotion (GT-based)
# -------------------------

rng = np.random.default_rng(MOVE_SELECTION_SEED)
ids_by_emotion = {emo: [img_id for img_id, y in zip(ids, labels) if y == emo] for emo in emotions}

for emo in emotions:
    if len(ids_by_emotion[emo]) < MOVES_PER_EMOTION:
        raise RuntimeError(f'Not enough images for emotion={emo}: have {len(ids_by_emotion[emo])}, need {MOVES_PER_EMOTION}')

sampled_ids = {
    emo: rng.choice(ids_by_emotion[emo], size=MOVES_PER_EMOTION, replace=False).tolist()
    for emo in emotions
}

# Move order interleaves emotions by round: round1(all emotions), round2(...), round3(...)
move_catalog = []
for r in range(MOVES_PER_EMOTION):
    for emo in emotions:
        move_catalog.append(sampled_ids[emo][r])

sample_table = []
for emo in emotions:
    for img_id in sampled_ids[emo]:
        sample_table.append({'sampled_from_emotion': emo, 'image_id': img_id})

print(f'total moves per axis = {len(move_catalog)}')
pd.DataFrame(sample_table).head(24)


### Gaussian Updates
This cell applies the Gaussian Bayesian axis updates across move steps and stores the resulting correlation matrices.


In [ ]:
# -------------------------
# Apply Gaussian Bayes moves and track correlation matrices at each move
# -------------------------

if AXIS_BAYES_MODE != 'gaussian':
    raise RuntimeError(f"This notebook currently implements gaussian updates only, got mode={AXIS_BAYES_MODE}")

corr_mats = []
axis_states_by_step = []

for step in range(len(move_catalog) + 1):
    axis_states = {}
    for axis_emo in emotions:
        # For this axis, each moved image gets target 1.0 if GT==axis, else 0.0
        moves = []
        for img_id in move_catalog[:step]:
            idx = id_to_idx[img_id]
            p01 = 1.0 if id_to_emotion.get(img_id, img_id.split('_', 1)[0].lower()) == axis_emo else 0.0
            moves.append((idx, p01))

        mu, b = gaussian_axis_update(
            X_all=X,
            w0=prior[axis_emo]['w0'],
            z0_sorted=prior[axis_emo]['z0_sorted'],
            moves=moves,
            alpha_vec=alpha_vec,
            bias_alpha=float(AXIS_BAYES_BIAS_ALPHA),
            sigma2=float(AXIS_BAYES_SIGMA2),
            b0=0.0,
        )
        axis_states[axis_emo] = {'mu': mu, 'b': b}

    C = compute_corr_matrix(axis_states)
    corr_mats.append(C)
    axis_states_by_step.append(axis_states)

print(f'Computed {len(corr_mats)} correlation matrices (from step 0 to step {len(corr_mats)-1}).')


### Quick Snapshots
This cell shows a few representative correlation matrices and the trajectory of the diagonal objective over time.


In [ ]:
# Quick snapshots
plot_corr_matrix(corr_mats[0], emotions, emotions, 'Step 0 (prior)')
plot_corr_matrix(corr_mats[1], emotions, emotions, 'Step 1')
plot_corr_matrix(corr_mats[len(corr_mats)//2], emotions, emotions, f'Step {len(corr_mats)//2}')
plot_corr_matrix(corr_mats[-1], emotions, emotions, f'Step {len(corr_mats)-1} (all moves)')
y = [sum(np.diag(C)) for C in corr_mats]
plt.figure(figsize=(8, 4))
plt.plot(y, marker='o')
plt.title('Sum of diagonal correlations (GT emotion vs same-axis projection)')


### Interactive Viewer
This cell adds an interactive slider so individual move steps can be inspected one at a time.


In [ ]:
# # Interactive: inspect correlation matrix at each move
# from ipywidgets import interact, IntSlider

# def show_step(step=0):
#     plot_corr_matrix(
#         corr_mats[step],
#         emotions,
#         emotions,
#         f'Step {step} / {len(corr_mats)-1}',
#     )

# interact(show_step, step=IntSlider(value=0, min=0, max=len(corr_mats)-1, step=1))


### Diagonal Trends
This cell plots the per-emotion diagonal correlations across all refinement steps.


In [ ]:
# Optional: track diagonal quality (axis emotion vs matching GT emotion correlation)
diag = np.array([np.diag(C) for C in corr_mats], dtype=np.float32)  # steps x emotions

plt.figure(figsize=(8, 4.5))
for j, emo in enumerate(emotions):
    plt.plot(diag[:, j], label=emo)
plt.axhline(0.0, color='k', linewidth=0.8)
plt.xlabel('Move step')
plt.ylabel('Correlation (axis vs matching GT)')
plt.title('Diagonal correlations across moves')
plt.legend(loc='best', fontsize=8, ncol=2)
plt.tight_layout()
plt.show()


## Extended Diagnostics, Qualitative Checks, and Hyperparameter Search

This section adds:
- mean uncertainty vs step,
- qualitative best/worst examples per axis over steps,
- standard Bayesian optimization diagnostics,
- parameter sweep + optimizer-driven hyperparameter search using external libraries.


### Extended Setup
This cell builds the extended experiment helpers used for uncertainty analysis, qualitative inspection, and hyperparameter search.


In [ ]:
from PIL import Image
from sklearn.model_selection import ParameterGrid

# Cache CLIP text directions once (emotion -> w_clip)
w_clip_text = {}
for emo in emotions:
    pos = l2_normalize_vec(encode_text(pos_prompts[emo]))
    neg = l2_normalize_vec(encode_text(neg_prompts[emo]))
    w_clip_text[emo] = l2_normalize_vec(pos - neg)


def build_features_and_priors(
    feature_space='clip_dino',
    clip_weight=0.70,
    dino_weight=0.30,
):
    fs = str(feature_space).strip().lower()
    if fs == 'clip_dino':
        if 'X_dino' not in globals():
            raise RuntimeError('X_dino not found. Load DINO embeddings first.')
        cw = float(max(1e-6, clip_weight))
        dw = float(max(1e-6, dino_weight))
        s = cw + dw
        cw = cw / s
        dw = dw / s
        clip_scale = float(np.sqrt(cw))
        dino_scale = float(np.sqrt(dw))
        X_curr = np.concatenate([clip_scale * X_clip, dino_scale * X_dino], axis=1).astype(np.float32)
        alpha_base = {
            'clip_dim': X_clip.shape[1],
            'dino_dim': X_dino.shape[1],
            'clip_scale': clip_scale,
            'dino_scale': dino_scale,
            'feature_space': 'clip_dino',
        }
        priors = {}
        for emo in emotions:
            w0 = l2_normalize_vec(np.concatenate([w_clip_text[emo], np.zeros((X_dino.shape[1],), dtype=np.float32)]))
            z0 = (X_curr @ w0).astype(np.float32)
            priors[emo] = {'w0': w0, 'b0': 0.0, 'z0': z0, 'z0_sorted': np.sort(z0).astype(np.float32)}
        return X_curr, priors, alpha_base

    if fs == 'clip':
        X_curr = X_clip.astype(np.float32)
        alpha_base = {
            'clip_dim': X_clip.shape[1],
            'dino_dim': 0,
            'clip_scale': 1.0,
            'dino_scale': 0.0,
            'feature_space': 'clip',
        }
        priors = {}
        for emo in emotions:
            w0 = w_clip_text[emo].copy().astype(np.float32)
            z0 = (X_curr @ w0).astype(np.float32)
            priors[emo] = {'w0': w0, 'b0': 0.0, 'z0': z0, 'z0_sorted': np.sort(z0).astype(np.float32)}
        return X_curr, priors, alpha_base

    raise ValueError(f'Unsupported feature_space: {feature_space}')


def gaussian_update_with_uncertainty(
    X_all,
    w0,
    z0_sorted,
    moves,
    alpha_vec,
    bias_alpha,
    sigma2,
    b0=0.0,
):
    w0 = np.asarray(w0, dtype=np.float32)
    alpha_vec = np.asarray(alpha_vec, dtype=np.float32)
    alpha_inv = 1.0 / np.maximum(alpha_vec, 1e-8)
    n = X_all.shape[0]

    # No-move posterior = prior
    if len(moves) == 0:
        mu = w0.copy()
        b = float(b0)
        prior_x = np.sum((X_all * X_all) * alpha_inv[None, :], axis=1).astype(np.float32)
        var = float(sigma2) + prior_x + (1.0 / float(bias_alpha))
        std = np.sqrt(np.maximum(var, 1e-8)).astype(np.float32)
        return mu, b, std

    idx = np.asarray([int(i) for i, _ in moves], dtype=np.int64)
    target_p01 = [float(p) for _, p in moves]
    X = X_all[idx, :].astype(np.float32)
    y_raw = np.asarray([quantile_from_sorted(z0_sorted, p) for p in target_p01], dtype=np.float32)

    m = X.shape[0]
    ones = np.ones((m,), dtype=np.float32)
    XS = (X * alpha_inv[None, :]).astype(np.float32)
    A = (
        (float(sigma2) * np.eye(m, dtype=np.float32))
        + (XS @ X.T)
        + ((1.0 / float(bias_alpha)) * np.outer(ones, ones).astype(np.float32))
    )
    A_inv = np.linalg.inv(A + (1e-6 * np.eye(m, dtype=np.float32)))

    r = y_raw - (X @ w0) - float(b0)
    mu = w0 + (alpha_inv * (X.T @ (A_inv @ r)))
    b = float(b0) + float((1.0 / float(bias_alpha)) * (ones @ (A_inv @ r)))

    if float(np.dot(mu, w0)) < 0.0:
        mu = -mu
        b = -b
    mu_n = float(np.linalg.norm(mu))
    if mu_n > 1e-8:
        mu = (mu / mu_n).astype(np.float32)
        b = float(b / mu_n)
    else:
        mu = w0.copy()
        b = float(b0)

    # Predictive std for every image
    prior_x = np.sum((X_all * X_all) * alpha_inv[None, :], axis=1).astype(np.float32)
    XS = (X * alpha_inv[None, :]).astype(np.float32)
    U = XS @ X_all.T  # m x n
    V = U + ((1.0 / float(bias_alpha)) * np.ones((m, 1), dtype=np.float32))
    AV = A_inv @ V
    quad = np.sum(V * AV, axis=0).astype(np.float32)
    var = float(sigma2) + prior_x + (1.0 / float(bias_alpha)) - quad
    std = np.sqrt(np.maximum(var, 1e-8)).astype(np.float32).reshape(n)
    return mu, b, std


def compute_corr_matrix_on_X(axis_state_dict, X_curr):
    rows = []
    for axis_emo in emotions:
        st = axis_state_dict[axis_emo]
        z = (X_curr @ st['mu']) + float(st['b'])
        rows.append([pearson_corr(z, Y[:, j]) for j in range(Y.shape[1])])
    return np.asarray(rows, dtype=np.float32)


def run_gaussian_experiment(
    feature_space='clip_dino',
    clip_weight=0.70,
    dino_weight=0.30,
    alpha=96.0,
    dino_alpha=220.0,
    bias_alpha=1.0,
    sigma2=0.04,
    move_catalog_override=None,
):
    if move_catalog_override is None:
        move_catalog_override = move_catalog

    X_curr, prior_curr, ab = build_features_and_priors(
        feature_space=feature_space,
        clip_weight=clip_weight,
        dino_weight=dino_weight,
    )

    if ab['feature_space'] == 'clip_dino':
        alpha_vec = np.concatenate([
            np.full((ab['clip_dim'],), float(alpha), dtype=np.float32),
            np.full((ab['dino_dim'],), float(dino_alpha), dtype=np.float32),
        ], axis=0)
    else:
        alpha_vec = np.full((ab['clip_dim'],), float(alpha), dtype=np.float32)

    corr_mats = []
    diag_sum = []
    axis_scores_by_step = []
    axis_std_by_step = []
    mean_uncertainty_global = []
    mean_uncertainty_per_axis = []

    n_steps = len(move_catalog_override) + 1
    for step in range(n_steps):
        axis_states = {}
        axis_scores = {}
        axis_stds = {}

        moved_ids = move_catalog_override[:step]
        for axis_emo in emotions:
            moves = []
            for img_id in moved_ids:
                idx = id_to_idx[img_id]
                gt = id_to_emotion.get(img_id, img_id.split('_', 1)[0].lower())
                target = 1.0 if gt == axis_emo else 0.0
                moves.append((idx, target))

            mu, b, std = gaussian_update_with_uncertainty(
                X_all=X_curr,
                w0=prior_curr[axis_emo]['w0'],
                z0_sorted=prior_curr[axis_emo]['z0_sorted'],
                moves=moves,
                alpha_vec=alpha_vec,
                bias_alpha=float(bias_alpha),
                sigma2=float(sigma2),
                b0=0.0,
            )
            z = (X_curr @ mu) + float(b)
            axis_states[axis_emo] = {'mu': mu, 'b': b}
            axis_scores[axis_emo] = z.astype(np.float32)
            axis_stds[axis_emo] = std.astype(np.float32)

        C = compute_corr_matrix_on_X(axis_states, X_curr)
        corr_mats.append(C)
        diag_sum.append(float(np.trace(C)))

        axis_scores_by_step.append(axis_scores)
        axis_std_by_step.append(axis_stds)

        per_axis_means = [float(np.mean(axis_stds[e])) for e in emotions]
        mean_uncertainty_per_axis.append(per_axis_means)
        mean_uncertainty_global.append(float(np.mean(per_axis_means)))

    out = {
        'X': X_curr,
        'feature_meta': ab,
        'alpha_vec': alpha_vec,
        'corr_mats': corr_mats,
        'diag_sum': np.asarray(diag_sum, dtype=np.float32),
        'axis_scores_by_step': axis_scores_by_step,
        'axis_std_by_step': axis_std_by_step,
        'mean_uncertainty_global': np.asarray(mean_uncertainty_global, dtype=np.float32),
        'mean_uncertainty_per_axis': np.asarray(mean_uncertainty_per_axis, dtype=np.float32),
    }
    return out


# Baseline run with current hyperparameters from the notebook top cell
baseline_ext = run_gaussian_experiment(
    feature_space=AXIS_BAYES_FEATURE_SPACE,
    clip_weight=AXIS_BAYES_CLIP_WEIGHT,
    dino_weight=AXIS_BAYES_DINO_WEIGHT,
    alpha=AXIS_BAYES_ALPHA,
    dino_alpha=AXIS_BAYES_DINO_ALPHA,
    bias_alpha=AXIS_BAYES_BIAS_ALPHA,
    sigma2=AXIS_BAYES_SIGMA2,
)
print('Extended baseline complete:', len(baseline_ext['corr_mats']), 'steps')


### Uncertainty Trajectory
This cell plots the average posterior uncertainty as a function of the number of user moves.


In [ ]:
# 1) Average uncertainty wrt steps

steps = np.arange(len(baseline_ext['diag_sum']))

plt.figure(figsize=(8, 4))
plt.plot(steps, baseline_ext['mean_uncertainty_global'], linewidth=2)
plt.xlabel('Step')
plt.ylabel('Mean uncertainty (global)')
plt.title('Average uncertainty vs step')
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4.5))
for j, emo in enumerate(emotions):
    plt.plot(steps, baseline_ext['mean_uncertainty_per_axis'][:, j], label=emo)
plt.xlabel('Step')
plt.ylabel('Mean uncertainty (per axis)')
plt.title('Per-axis mean uncertainty vs step')
plt.legend(loc='best', fontsize=8, ncol=2)
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()


### Optimization Diagnostics
This cell visualizes standard Bayesian optimization diagnostics such as objective trajectories, heatmaps, and score-versus-uncertainty plots.


In [ ]:
# 2) Typical diagnostics during Bayesian optimization

# (a) Objective trajectory: sum of diagonal correlations
plt.figure(figsize=(8, 4))
plt.plot(baseline_ext['diag_sum'], linewidth=2)
plt.xlabel('Step')
plt.ylabel('Sum diag(corr)')
plt.title('Optimization objective vs step')
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

# (b) Heatmap of diagonal terms over time (axis-specific quality)
diag_mat = np.array([np.diag(C) for C in baseline_ext['corr_mats']], dtype=np.float32)  # steps x emotions
plt.figure(figsize=(9, 4.5))
plt.imshow(diag_mat.T, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
plt.yticks(np.arange(len(emotions)), emotions)
plt.xlabel('Step')
plt.title('Diagonal correlation per axis over steps')
plt.colorbar(fraction=0.03, pad=0.02)
plt.tight_layout()
plt.show()

# (c) Score distribution + prior/final agreement for one axis
axis_to_view = emotions[0]
z_prior = baseline_ext['axis_scores_by_step'][0][axis_to_view]
z_final = baseline_ext['axis_scores_by_step'][-1][axis_to_view]
std_final = baseline_ext['axis_std_by_step'][-1][axis_to_view]

fig, axs = plt.subplots(1, 3, figsize=(15, 4.2))
axs[0].hist(z_prior, bins=24, alpha=0.6, label='prior')
axs[0].hist(z_final, bins=24, alpha=0.6, label='final')
axs[0].set_title(f'{axis_to_view}: score distribution')
axs[0].legend()

axs[1].scatter(z_prior, z_final, s=10, alpha=0.6)
mn = float(min(z_prior.min(), z_final.min()))
mx = float(max(z_prior.max(), z_final.max()))
axs[1].plot([mn, mx], [mn, mx], 'k--', linewidth=1)
axs[1].set_title(f'{axis_to_view}: prior vs final scores')
axs[1].set_xlabel('prior')
axs[1].set_ylabel('final')

axs[2].scatter(z_final, std_final, s=10, alpha=0.6)
axs[2].set_title(f'{axis_to_view}: uncertainty vs score (final)')
axs[2].set_xlabel('score')
axs[2].set_ylabel('std')

plt.tight_layout()
plt.show()


### Qualitative Extremes
This cell displays the lowest- and highest-scoring EmoSet examples for each axis across selected refinement steps.


In [ ]:
# 3) Qualitative examples: best/worst per axis over steps

def _load_img(image_id):
    path = dataset_root / image_id
    with Image.open(path) as im:
        return im.convert('RGB').copy()


def show_axis_extremes_over_steps(results, axis_emo, steps_to_show=None, k=3):
    if steps_to_show is None:
        steps_to_show = [0, len(results['diag_sum']) // 2, len(results['diag_sum']) - 1]

    n_rows = len(steps_to_show)
    n_cols = 2 * k
    fig, axs = plt.subplots(n_rows, n_cols, figsize=(2.3 * n_cols, 2.5 * n_rows))
    if n_rows == 1:
        axs = np.expand_dims(axs, axis=0)

    for r, step in enumerate(steps_to_show):
        scores = results['axis_scores_by_step'][step][axis_emo]
        stds = results['axis_std_by_step'][step][axis_emo]
        order = np.argsort(scores)
        worst = order[:k]
        best = order[-k:][::-1]
        idxs = list(worst) + list(best)

        for c, idx in enumerate(idxs):
            ax = axs[r, c]
            img_id = ids[int(idx)]
            img = _load_img(img_id)
            ax.imshow(img)
            ax.axis('off')
            side = 'worst' if c < k else 'best'
            ax.set_title(
                f'step {step} | {side}\n{img_id}\nscore={scores[idx]:.3f} std={stds[idx]:.3f}',
                fontsize=7,
            )

    fig.suptitle(f'Axis: {axis_emo} (left: worst, right: best)', fontsize=12)
    plt.tight_layout()
    plt.show()


# Show for every axis (change k if you want more)
for emo in emotions:
    show_axis_extremes_over_steps(baseline_ext, emo, k=3)


### Grid Sweep
This cell runs a small hyperparameter sweep and compares the resulting objective trajectories and final scores.


In [ ]:
# 4) Parameter sweep (library-based) and trajectory tracking

# Keep this moderate first; increase values once stable.
SWEEP_GRID = {
    'feature_space': [AXIS_BAYES_FEATURE_SPACE],
    'clip_weight': [0.6, 0.7, 0.8],
    'dino_weight': [0.4, 0.3, 0.2],
    'alpha': [64.0, 96.0, 160.0],
    'dino_alpha': [160.0, 220.0, 320.0],
    'bias_alpha': [0.5, 1.0, 2.0],
    'sigma2': [0.02, 0.04, 0.08],
}

# If you want to reduce compute, set MAX_SWEEP_COMBOS to a smaller number
MAX_SWEEP_COMBOS = 60

all_cfgs = list(ParameterGrid(SWEEP_GRID))
if len(all_cfgs) > MAX_SWEEP_COMBOS:
    rng = np.random.default_rng(2026)
    chosen = rng.choice(len(all_cfgs), size=MAX_SWEEP_COMBOS, replace=False)
    cfgs = [all_cfgs[int(i)] for i in chosen]
else:
    cfgs = all_cfgs

print(f'Trying {len(cfgs)} configs (from total={len(all_cfgs)})')

sweep_rows = []
traj_by_cfg = {}
for i, cfg in enumerate(cfgs, 1):
    res = run_gaussian_experiment(**cfg)
    traj = res['diag_sum']
    cfg_id = f'cfg_{i:03d}'
    traj_by_cfg[cfg_id] = traj.copy()
    sweep_rows.append({
        'cfg_id': cfg_id,
        **cfg,
        'final_diag_sum': float(traj[-1]),
        'best_step_diag_sum': float(np.max(traj)),
        'auc_diag_sum': float(np.mean(traj)),
        'final_mean_uncertainty': float(res['mean_uncertainty_global'][-1]),
    })
    if i % 10 == 0 or i == len(cfgs):
        print(f'  done {i}/{len(cfgs)}')

sweep_df = pd.DataFrame(sweep_rows).sort_values('final_diag_sum', ascending=False).reset_index(drop=True)
display(sweep_df.head(15))

# Plot top trajectory curves
top_n = min(8, len(sweep_df))
plt.figure(figsize=(9, 4.5))
for _, row in sweep_df.head(top_n).iterrows():
    cfg_id = row['cfg_id']
    plt.plot(traj_by_cfg[cfg_id], label=f"{cfg_id} final={row['final_diag_sum']:.2f}")
plt.xlabel('Step')
plt.ylabel('Sum diag(corr)')
plt.title('Top hyperparameter trajectories')
plt.legend(fontsize=7, ncol=2)
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

best_cfg = sweep_df.iloc[0].to_dict()
print('Best config by final_diag_sum:')
for k, v in best_cfg.items():
    if k == 'cfg_id' or isinstance(v, str):
        print(f'  {k}: {v}')
    else:
        print(f'  {k}: {float(v):.6f}')


### Parallel Coordinates
This cell loads the Optuna CSV log and plots the sweep parameters as a parallel-coordinate view colored by the chosen objective.


In [ ]:
# 4b) Parallel coordinates from Optuna CSV log (standalone)
# Reads the append-only CSV produced by backend/optuna_emoset_sweep.py

CSV_PATH = REPO_ROOT / 'backend' / 'experiments' / 'emoset_optuna_runs.csv'
PARALLEL_COLOR_METRIC = 'final_diag_sum'  # or: 'value', 'auc_diag_sum', 'best_step_diag_sum'
PARALLEL_PARAM_COLS = ['clip_weight', 'dino_weight', 'alpha', 'dino_alpha', 'bias_alpha', 'sigma2']

if not CSV_PATH.exists():
    raise FileNotFoundError(f'CSV log not found: {CSV_PATH}')

try:
    import plotly.express as px
except Exception as exc:
    raise RuntimeError('plotly is required for this cell. Install with: pip install plotly') from exc

csv_df = pd.read_csv(CSV_PATH)
# Keep only successful trial rows
csv_df = csv_df[(csv_df['row_type'] == 'trial') & (csv_df['trial_state'] == 'ok')].copy()
if len(csv_df) == 0:
    raise RuntimeError(f'No successful trial rows found in {CSV_PATH}')

for c in PARALLEL_PARAM_COLS + [PARALLEL_COLOR_METRIC]:
    csv_df[c] = pd.to_numeric(csv_df[c], errors='coerce')
csv_df = csv_df.dropna(subset=PARALLEL_PARAM_COLS + [PARALLEL_COLOR_METRIC]).reset_index(drop=True)

labels = {
    'clip_weight': 'clip_weight',
    'dino_weight': 'dino_weight',
    'alpha': 'alpha',
    'dino_alpha': 'dino_alpha',
    'bias_alpha': 'bias_alpha',
    'sigma2': 'sigma2',
    'value': 'objective (value)',
    'final_diag_sum': 'objective (final_diag_sum)',
    'auc_diag_sum': 'objective (auc_diag_sum)',
    'best_step_diag_sum': 'objective (best_step_diag_sum)',
}

fig = px.parallel_coordinates(
    csv_df,
    dimensions=PARALLEL_PARAM_COLS,
    color=PARALLEL_COLOR_METRIC,
    color_continuous_scale=px.colors.sequential.Viridis,
    labels=labels,
)
fig.update_layout(
    title=f'Optuna Trials from CSV (color = {PARALLEL_COLOR_METRIC})',
    width=1100,
    height=550,
)
fig.show()


### Rank Sweep Parallel Coordinates
This cell loads the rank Optuna CSV and plots the rank hyperparameters as a parallel-coordinate view colored by the final macro-AUROC objective.


In [ ]:
# 4c) Parallel coordinates for the rank Optuna sweep

RANK_CSV_PATH = REPO_ROOT / 'backend' / 'experiments' / 'emoset_optuna_rank.csv'
RANK_PARALLEL_COLOR_METRIC = 'final_macro_auroc'  # or: 'value', 'auc_macro_auroc', 'final_diag_sum'
RANK_PARALLEL_PARAM_COLS = [
    'clip_weight',
    'dino_weight',
    'alpha',
    'dino_alpha',
    'rank_eta',
    'rank_anchor_k',
    'rank_anchor_delta',
    'rank_max_pairs',
]

if not RANK_CSV_PATH.exists():
    raise FileNotFoundError(f'Rank CSV log not found: {RANK_CSV_PATH}')

try:
    import plotly.express as px
except Exception as exc:
    raise RuntimeError('plotly is required for this cell. Install with: pip install plotly') from exc

rank_df = pd.read_csv(RANK_CSV_PATH)
rank_df = rank_df[(rank_df['row_type'] == 'trial') & (rank_df['trial_state'] == 'ok')].copy()
if 'mode' in rank_df.columns:
    rank_df = rank_df[rank_df['mode'].astype(str).str.lower() == 'rank'].copy()
if len(rank_df) == 0:
    raise RuntimeError(f'No successful rank trial rows found in {RANK_CSV_PATH}')

for c in RANK_PARALLEL_PARAM_COLS + [RANK_PARALLEL_COLOR_METRIC]:
    rank_df[c] = pd.to_numeric(rank_df[c], errors='coerce')
rank_df = rank_df.dropna(subset=RANK_PARALLEL_PARAM_COLS + [RANK_PARALLEL_COLOR_METRIC]).reset_index(drop=True)

rank_labels = {
    'clip_weight': 'clip_weight',
    'dino_weight': 'dino_weight',
    'alpha': 'alpha',
    'dino_alpha': 'dino_alpha',
    'rank_eta': 'rank_eta',
    'rank_anchor_k': 'rank_anchor_k',
    'rank_anchor_delta': 'rank_anchor_delta',
    'rank_max_pairs': 'rank_max_pairs',
    'value': 'objective (value)',
    'final_macro_auroc': 'objective (final_macro_auroc)',
    'auc_macro_auroc': 'objective (auc_macro_auroc)',
    'final_diag_sum': 'diag objective (final_diag_sum)',
}

fig = px.parallel_coordinates(
    rank_df,
    dimensions=RANK_PARALLEL_PARAM_COLS,
    color=RANK_PARALLEL_COLOR_METRIC,
    color_continuous_scale=px.colors.sequential.Viridis,
    labels=rank_labels,
)
fig.update_layout(
    title=f'Rank Optuna Trials (color = {RANK_PARALLEL_COLOR_METRIC})',
    width=1200,
    height=560,
)
fig.show()


### Optuna Search
This cell provides an optional Optuna-based search that maximizes the final EmoSet diagonal-correlation objective.


In [ ]:
# 5) Optional: Optuna search to maximize final accuracy (final sum diagonal)
# Requires: pip install optuna

USE_OPTUNA = False
OPTUNA_TRIALS = 40

if USE_OPTUNA:
    import optuna

    def objective(trial):
        cfg = {
            'feature_space': AXIS_BAYES_FEATURE_SPACE,
            'clip_weight': trial.suggest_float('clip_weight', 0.1, 0.9),
            'dino_weight': trial.suggest_float('dino_weight', 0.1, 0.9),
            'alpha': trial.suggest_float('alpha', 1.0, 220.0, log=True),
            'dino_alpha': trial.suggest_float('dino_alpha', 1.0, 520.0, log=True),
            'bias_alpha': trial.suggest_float('bias_alpha', 0.2, 4.0, log=True),
            'sigma2': trial.suggest_float('sigma2', 0.005, 1.0, log=True),
        }
        res = run_gaussian_experiment(**cfg)
        return float(res['diag_sum'][-1])

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)

    print('Best Optuna objective (final_diag_sum):', study.best_value)
    print('Best params:')
    for k, v in study.best_params.items():
        print(f'  {k}: {v}')

    # Evaluate full trajectory for best params
    best_opt_res = run_gaussian_experiment(**study.best_params, feature_space=AXIS_BAYES_FEATURE_SPACE)
    plt.figure(figsize=(8, 4))
    plt.plot(best_opt_res['diag_sum'], linewidth=2)
    plt.xlabel('Step')
    plt.ylabel('Sum diag(corr)')
    plt.title('Best Optuna trajectory')
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()
else:
    print('Set USE_OPTUNA=True to run Optuna search.')


## Graph

This section evaluates the **graph Bayesian field** on EmoSet using the same move protocol as above.

It keeps the same text-derived prior scores `z0`, but replaces the global linear posterior with a Gaussian random field on a kNN graph over the image embeddings:
- prior mean = `z0`
- prior precision = `lambda_smooth * L + lambda_prior * I`
- moved images become noisy observed nodes
- posterior mean gives the refined axis score
- posterior std gives uncertainty per image


### Graph Setup
This cell builds the graph-mode helpers, runs the EmoSet graph Bayesian field baseline, and caches its outputs.


In [ ]:
# Graph-mode helpers and baseline EmoSet run

if 'build_features_and_priors' not in globals():
    raise RuntimeError('Run the earlier setup/extended diagnostics cells first (build_features_and_priors is missing).')


def build_graph_prior_on_X(X_all, knn_k=16, lambda_smooth=6.0, lambda_prior=1.0, jitter=1e-6):
    X_all = np.asarray(X_all, dtype=np.float32)
    n = int(X_all.shape[0])
    if n <= 0:
        raise RuntimeError('Cannot build graph prior for empty feature matrix')
    if n == 1:
        q0 = np.asarray([[float(lambda_prior) + float(jitter)]], dtype=np.float64)
        k0 = np.asarray([[1.0 / q0[0, 0]]], dtype=np.float32)
        return {
            'K0': k0,
            'diag': np.asarray([float(k0[0, 0])], dtype=np.float32),
            'knn_k': 0,
            'scale': 1.0,
        }

    k = min(max(1, int(knn_k)), n - 1)
    sim = np.asarray(X_all @ X_all.T, dtype=np.float32)
    np.fill_diagonal(sim, -np.inf)
    kth = max(0, k - 1)
    nbr_idx = np.argpartition(-sim, kth=kth, axis=1)[:, :k]
    row_idx = np.arange(n, dtype=np.int64)[:, None]
    nbr_sim = np.asarray(sim[row_idx, nbr_idx], dtype=np.float32)
    nbr_dist = np.clip(1.0 - np.clip(nbr_sim, -1.0, 1.0), 0.0, 2.0).astype(np.float32)
    scale = float(np.mean(nbr_dist)) if nbr_dist.size > 0 else 1.0
    scale = max(scale, 1e-3)
    nbr_w = np.exp(-nbr_dist / scale).astype(np.float32)

    W = np.zeros((n, n), dtype=np.float32)
    W[row_idx, nbr_idx] = nbr_w
    W = np.maximum(W, W.T).astype(np.float32)
    np.fill_diagonal(W, 0.0)

    degree = np.sum(W, axis=1).astype(np.float32)
    L = (np.diag(degree) - W).astype(np.float32)
    q0 = (
        (float(lambda_smooth) * L.astype(np.float64))
        + ((float(lambda_prior) + float(jitter)) * np.eye(n, dtype=np.float64))
    )
    try:
        k0 = np.linalg.inv(q0).astype(np.float32)
    except Exception:
        k0 = np.linalg.pinv(q0).astype(np.float32)
    return {
        'K0': k0,
        'diag': np.diag(k0).astype(np.float32),
        'knn_k': int(k),
        'scale': float(scale),
    }


def graph_axis_update_with_uncertainty(z0, z0_sorted, prior_cov, prior_cov_diag, moves, sigma2):
    z0 = np.asarray(z0, dtype=np.float32).reshape(-1)
    prior_cov = np.asarray(prior_cov, dtype=np.float32)
    prior_cov_diag = np.asarray(prior_cov_diag, dtype=np.float32).reshape(-1)
    n = int(z0.size)

    if len(moves) == 0:
        std = np.sqrt(np.maximum(prior_cov_diag, 1e-8)).astype(np.float32)
        return z0.copy(), std

    obs_idx = np.asarray([int(i) for i, _ in moves], dtype=np.int64)
    target_p01 = [float(p) for _, p in moves]
    y_raw = np.asarray([quantile_from_sorted(z0_sorted, p) for p in target_p01], dtype=np.float32)
    obs_var = np.full((len(moves),), float(sigma2), dtype=np.float32)

    K_xo = prior_cov[:, obs_idx]
    K_oo = prior_cov[np.ix_(obs_idx, obs_idx)]
    S = np.asarray(K_oo + np.diag(obs_var), dtype=np.float64)
    try:
        S_inv = np.linalg.inv(S + (1e-6 * np.eye(S.shape[0], dtype=np.float64))).astype(np.float32)
    except Exception:
        S_inv = np.linalg.pinv(S + (1e-6 * np.eye(S.shape[0], dtype=np.float64))).astype(np.float32)

    residual = np.asarray(y_raw - z0[obs_idx], dtype=np.float32)
    gain = np.asarray(S_inv @ residual, dtype=np.float32)
    post_mean = np.asarray(z0 + (K_xo @ gain), dtype=np.float32)

    tmp = np.asarray(S_inv @ K_xo.T, dtype=np.float32)
    quad = np.sum(K_xo * tmp.T, axis=1).astype(np.float32)
    post_var = np.asarray(prior_cov_diag - quad, dtype=np.float32)
    std = np.sqrt(np.maximum(post_var, 1e-8)).astype(np.float32).reshape(n)
    return post_mean, std


def compute_corr_matrix_from_scores(axis_score_dict, X_curr=None):
    rows = []
    for axis_emo in emotions:
        z = np.asarray(axis_score_dict[axis_emo], dtype=np.float32)
        rows.append([pearson_corr(z, Y[:, j]) for j in range(Y.shape[1])])
    return np.asarray(rows, dtype=np.float32)


def run_graph_experiment(
    feature_space='clip_dino',
    clip_weight=0.70,
    dino_weight=0.30,
    graph_knn_k=16,
    graph_lambda_smooth=6.0,
    graph_lambda_prior=1.0,
    graph_jitter=1e-6,
    sigma2=0.04,
    move_catalog_override=None,
):
    if move_catalog_override is None:
        move_catalog_override = move_catalog

    X_curr, prior_curr, feature_meta = build_features_and_priors(
        feature_space=feature_space,
        clip_weight=clip_weight,
        dino_weight=dino_weight,
    )
    graph_prior = build_graph_prior_on_X(
        X_curr,
        knn_k=graph_knn_k,
        lambda_smooth=graph_lambda_smooth,
        lambda_prior=graph_lambda_prior,
        jitter=graph_jitter,
    )

    corr_mats = []
    diag_sum = []
    axis_scores_by_step = []
    axis_std_by_step = []
    mean_uncertainty_global = []
    mean_uncertainty_per_axis = []

    n_steps = len(move_catalog_override) + 1
    for step in range(n_steps):
        axis_scores = {}
        axis_stds = {}
        moved_ids = move_catalog_override[:step]

        for axis_emo in emotions:
            moves = []
            for img_id in moved_ids:
                idx = id_to_idx[img_id]
                gt = id_to_emotion.get(img_id, img_id.split('_', 1)[0].lower())
                target = 1.0 if gt == axis_emo else 0.0
                moves.append((idx, target))

            z, std = graph_axis_update_with_uncertainty(
                z0=prior_curr[axis_emo]['z0'],
                z0_sorted=prior_curr[axis_emo]['z0_sorted'],
                prior_cov=graph_prior['K0'],
                prior_cov_diag=graph_prior['diag'],
                moves=moves,
                sigma2=float(sigma2),
            )
            axis_scores[axis_emo] = z.astype(np.float32)
            axis_stds[axis_emo] = std.astype(np.float32)

        C = compute_corr_matrix_from_scores(axis_scores, X_curr)
        corr_mats.append(C)
        diag_sum.append(float(np.trace(C)))
        axis_scores_by_step.append(axis_scores)
        axis_std_by_step.append(axis_stds)

        per_axis_means = [float(np.mean(axis_stds[e])) for e in emotions]
        mean_uncertainty_per_axis.append(per_axis_means)
        mean_uncertainty_global.append(float(np.mean(per_axis_means)))

    return {
        'X': X_curr,
        'feature_meta': feature_meta,
        'graph_prior': graph_prior,
        'corr_mats': corr_mats,
        'diag_sum': np.asarray(diag_sum, dtype=np.float32),
        'axis_scores_by_step': axis_scores_by_step,
        'axis_std_by_step': axis_std_by_step,
        'mean_uncertainty_global': np.asarray(mean_uncertainty_global, dtype=np.float32),
        'mean_uncertainty_per_axis': np.asarray(mean_uncertainty_per_axis, dtype=np.float32),
    }


graph_baseline = run_graph_experiment(
    feature_space=AXIS_BAYES_FEATURE_SPACE,
    clip_weight=AXIS_BAYES_CLIP_WEIGHT,
    dino_weight=AXIS_BAYES_DINO_WEIGHT,
    graph_knn_k=AXIS_BAYES_GRAPH_KNN_K,
    graph_lambda_smooth=AXIS_BAYES_GRAPH_LAMBDA_SMOOTH,
    graph_lambda_prior=AXIS_BAYES_GRAPH_LAMBDA_PRIOR,
    graph_jitter=AXIS_BAYES_GRAPH_JITTER,
    sigma2=AXIS_BAYES_SIGMA2,
)
print('Graph baseline complete:', len(graph_baseline['corr_mats']), 'steps')
print('Graph kNN:', graph_baseline['graph_prior']['knn_k'], 'scale:', graph_baseline['graph_prior']['scale'])


### Graph Diagnostics
This cell visualizes the graph-mode correlation matrices, trajectories, uncertainties, and qualitative examples.


In [ ]:
# Graph-mode diagnostics on EmoSet

plot_corr_matrix(graph_baseline['corr_mats'][0], emotions, emotions, 'Graph prior correlation matrix')
plot_corr_matrix(graph_baseline['corr_mats'][-1], emotions, emotions, 'Graph final correlation matrix')

plt.figure(figsize=(8, 4.2))
plt.plot(graph_baseline['diag_sum'], linewidth=2, label='graph')
if 'baseline_ext' in globals():
    plt.plot(baseline_ext['diag_sum'], linewidth=2, linestyle='--', label='gaussian')
plt.xlabel('Step')
plt.ylabel('Sum diag(corr)')
plt.title('Graph trajectory on EmoSet')
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4.2))
plt.plot(graph_baseline['mean_uncertainty_global'], linewidth=2, color='tab:orange')
plt.xlabel('Step')
plt.ylabel('Mean uncertainty')
plt.title('Graph mean uncertainty vs step')
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

axis_to_view = emotions[0]
z_prior = graph_baseline['axis_scores_by_step'][0][axis_to_view]
z_final = graph_baseline['axis_scores_by_step'][-1][axis_to_view]
std_final = graph_baseline['axis_std_by_step'][-1][axis_to_view]

fig, axs = plt.subplots(1, 3, figsize=(15, 4.2))
axs[0].hist(z_prior, bins=24, alpha=0.6, label='prior')
axs[0].hist(z_final, bins=24, alpha=0.6, label='final')
axs[0].set_title(f'{axis_to_view}: graph score distribution')
axs[0].legend()

axs[1].scatter(z_prior, z_final, s=10, alpha=0.6)
mn = float(min(z_prior.min(), z_final.min()))
mx = float(max(z_prior.max(), z_final.max()))
axs[1].plot([mn, mx], [mn, mx], 'k--', linewidth=1)
axs[1].set_title(f'{axis_to_view}: graph prior vs final')
axs[1].set_xlabel('prior')
axs[1].set_ylabel('final')

axs[2].scatter(z_final, std_final, s=10, alpha=0.6)
axs[2].set_title(f'{axis_to_view}: graph uncertainty vs score')
axs[2].set_xlabel('score')
axs[2].set_ylabel('std')

plt.tight_layout()
plt.show()

show_axis_extremes_over_steps(graph_baseline, axis_to_view, k=3)


## Rank Sweep CSV

This section loads the saved rank Optuna sweep log and visualizes the hyperparameter landscape with a parallel-coordinates plot colored by the final macro-AUROC objective.


In [ ]:
import pandas as pd
import plotly.express as px

rank_csv = REPO_ROOT / 'backend' / 'experiments' / 'emoset_optuna_rank.csv'
rank_color_metric = 'final_macro_auroc'
rank_parallel_cols = [
    'clip_weight',
    'dino_weight',
    'alpha',
    'dino_alpha',
    'rank_eta',
    'rank_anchor_k',
    'rank_anchor_delta',
    'rank_max_pairs',
]

if not rank_csv.exists():
    raise FileNotFoundError(f'Missing rank sweep CSV: {rank_csv}')

rank_trials = pd.read_csv(rank_csv)
if 'row_type' in rank_trials.columns:
    rank_trials = rank_trials[rank_trials['row_type'].astype(str) == 'trial']
if 'trial_state' in rank_trials.columns:
    rank_trials = rank_trials[rank_trials['trial_state'].astype(str).str.lower() == 'ok']
if 'mode' in rank_trials.columns:
    rank_trials = rank_trials[rank_trials['mode'].astype(str).str.lower() == 'rank']
if rank_trials.empty:
    raise RuntimeError(f'No successful rank trials found in {rank_csv.name}')

for col in rank_parallel_cols + [rank_color_metric]:
    rank_trials[col] = pd.to_numeric(rank_trials[col], errors='coerce')
rank_trials = rank_trials.dropna(subset=rank_parallel_cols + [rank_color_metric]).reset_index(drop=True)

rank_labels = {
    'clip_weight': 'clip_weight',
    'dino_weight': 'dino_weight',
    'alpha': 'alpha',
    'dino_alpha': 'dino_alpha',
    'rank_eta': 'rank_eta',
    'rank_anchor_k': 'rank_anchor_k',
    'rank_anchor_delta': 'rank_anchor_delta',
    'rank_max_pairs': 'rank_max_pairs',
    'value': 'objective (value)',
    'final_macro_auroc': 'objective (final_macro_auroc)',
    'auc_macro_auroc': 'objective (auc_macro_auroc)',
    'final_diag_sum': 'diag objective (final_diag_sum)',
}

rank_fig = px.parallel_coordinates(
    rank_trials,
    dimensions=rank_parallel_cols,
    color=rank_color_metric,
    color_continuous_scale=px.colors.sequential.Viridis,
    labels=rank_labels,
)
rank_fig.update_layout(
    title=f'Rank Optuna Sweep (color = {rank_color_metric})',
    width=1200,
    height=560,
)
rank_fig.show()


## Piecewise Sweep CSV

This section loads the piecewise multidataset Optuna sweep log and visualizes the hyperparameter landscape from the saved CSV.


### Load Piecewise Sweep CSV
This cell loads the piecewise multidataset sweep CSV, keeps successful trials, and shows the top runs.


In [11]:
from pathlib import Path
import json
import pandas as pd

piecewise_csv = Path('/home/mario/codes/promptherder/backend/experiments/piecewise_multidataset_optuna.csv')
if not piecewise_csv.exists():
    raise FileNotFoundError(f'Missing sweep CSV: {piecewise_csv}')

piecewise_df = pd.read_csv(piecewise_csv)
piecewise_trials = piecewise_df.copy()
if 'row_type' in piecewise_trials.columns:
    piecewise_trials = piecewise_trials[piecewise_trials['row_type'].astype(str) == 'trial']
if 'trial_state' in piecewise_trials.columns:
    piecewise_trials = piecewise_trials[piecewise_trials['trial_state'].astype(str).str.lower() == 'ok']
piecewise_trials = piecewise_trials.dropna(subset=['value']).sort_values('value', ascending=False).reset_index(drop=True)
piecewise_trials['emoset_emotion'] = piecewise_trials['per_task_diag_mean_json'].apply(lambda x: json.loads(x).get('emoset_emotion'))
piecewise_trials['paintings_genre'] = piecewise_trials['per_task_diag_mean_json'].apply(lambda x: json.loads(x).get('paintings_wikiart_genre'))
piecewise_trials['paintings_style'] = piecewise_trials['per_task_diag_mean_json'].apply(lambda x: json.loads(x).get('paintings_wikiart_style'))

display(piecewise_trials[['trial_number', 'value', 'clip_weight', 'dino_weight', 'piecewise_num_experts', 'piecewise_use_gating', 'piecewise_aggregator','emoset_emotion', 'paintings_genre', 'paintings_style']].head(10))
print(f'Loaded {len(piecewise_trials)} successful trials from {piecewise_csv.name}')


,trial_number,value,clip_weight,dino_weight,piecewise_num_experts,piecewise_use_gating,piecewise_aggregator,emoset_emotion,paintings_genre,paintings_style
0,85,0.451442,0.632810,0.367190,1,True,max,0.477213,0.529143,0.347971
1,82,0.451213,0.647639,0.352361,1,True,max,0.489077,0.524314,0.340248
2,61,0.450779,0.566818,0.433182,1,True,max,0.480476,0.528009,0.343853
3,42,0.449600,0.436362,0.563638,1,True,max,0.487274,0.523385,0.338141
4,51,0.449306,0.594078,0.405922,1,True,max,0.490769,0.521220,0.335928
5,113,0.448924,0.483873,0.516127,1,True,max,0.478119,0.526361,0.342291
6,53,0.448842,0.582580,0.417420,1,True,max,0.474170,0.531410,0.340946
7,84,0.448529,0.590327,0.409673,1,True,max,0.465988,0.530488,0.349112
8,69,0.448282,0.575873,0.424127,1,True,max,0.473307,0.527296,0.344244
9,94,0.447721,0.578500,0.421500,1,True,max,0.461114,0.532428,0.349619


Loaded 120 successful trials from piecewise_multidataset_optuna.csv


In [ ]:
import json

s = '{"emoset_emotion":3.483581066131592,"paintings_wikiart_genre":4.618589878082275,"paintings_wikiart_style":4.039992332458496}'
d = json.loads(s)
d

'{"emoset_emotion":3.483581066131592,"paintings_wikiart_genre":4.618589878082275,"paintings_wikiart_style":4.039992332458496}'

### Parallel Coordinates Plot
This cell renders a parallel plot of the main piecewise hyperparameters, colored by the final sweep objective.


In [12]:
import pandas as pd
import plotly.graph_objects as go

parallel_cols = [
    'clip_weight',
    'dino_weight',
    'piecewise_num_experts',
    'piecewise_use_gating',
    'piecewise_aggregator',
    'piecewise_clip_scale',
    'piecewise_dino_scale',
    'pairwise_from_scalar_margin',
    'piecewise_prior_strength',
    'piecewise_expert_diversity_strength',
    'piecewise_l2_reg',
    'piecewise_learning_rate',
    'piecewise_max_refine_steps',
]

parallel_dims = []
for col in parallel_cols:
    if col not in piecewise_trials.columns:
        continue
    s = piecewise_trials[col]
    label = col.replace('piecewise_', 'pw_').replace('pairwise_from_scalar_margin', 'pair_margin')
    if pd.api.types.is_bool_dtype(s):
        parallel_dims.append(dict(label=label, values=s.astype(int), tickvals=[0, 1], ticktext=['False', 'True']))
    elif pd.api.types.is_numeric_dtype(s):
        parallel_dims.append(dict(label=label, values=s.astype(float)))
    else:
        categories = list(pd.Index(s.astype(str)).dropna().unique())
        mapping = {cat: idx for idx, cat in enumerate(categories)}
        parallel_dims.append(
            dict(
                label=label,
                values=s.astype(str).map(mapping),
                tickvals=list(mapping.values()),
                ticktext=list(mapping.keys()),
            )
        )

fig = go.Figure(
    data=go.Parcoords(
        line=dict(
            color=piecewise_trials['emoset_emotion'].astype(float),
            colorscale='Viridis',
            showscale=True,
            colorbar=dict(title='emoset_emotion'),
        ),
        dimensions=parallel_dims,
    )
)
fig.update_layout(height=560, title='Piecewise multidataset Optuna sweep')
fig.show()
fig = go.Figure(
    data=go.Parcoords(
        line=dict(
            color=piecewise_trials['paintings_genre'].astype(float),
            colorscale='Viridis',
            showscale=True,
            colorbar=dict(title='paintings_genre'),
        ),
        dimensions=parallel_dims,
    )
)
fig.update_layout(height=560, title='Piecewise multidataset Optuna sweep')
fig.show()

fig = go.Figure(
    data=go.Parcoords(
        line=dict(
            color=piecewise_trials['paintings_style'].astype(float),
            colorscale='Viridis',
            showscale=True,
            colorbar=dict(title='paintings_style'),
        ),
        dimensions=parallel_dims,
    )
)
fig.update_layout(height=560, title='Piecewise multidataset Optuna sweep')
fig.show()


## Modeling Evaluation


### Load Modeling Evaluation CSVs
This cell loads the modeling-evaluation CSV outputs and selects the run to analyze.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

MODELING_DIR = Path('/home/mario/codes/promptherder/backend/experiments/modeling_eval')
MODELING_RUN_ID = None  # set to a specific run_id string to override the latest run


def load_eval_csv(name: str) -> pd.DataFrame:
    path = MODELING_DIR / f'{name}.csv'
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_csv(path)


runs_df = load_eval_csv('runs')
prior_df = load_eval_csv('prior')
refinement_df = load_eval_csv('refinement')
query_policy_df = load_eval_csv('query_policy')
uncertainty_df = load_eval_csv('uncertainty')
undefined_df = load_eval_csv('undefined')
ablation_df = load_eval_csv('ablation')
tasks_df = load_eval_csv('tasks')

if runs_df.empty:
    raise RuntimeError('No modeling evaluation runs found.')

runs_df = runs_df.sort_values('ts_utc').reset_index(drop=True)
selected_run_id = MODELING_RUN_ID or str(runs_df.iloc[-1]['run_id'])
selected_run = runs_df[runs_df['run_id'].astype(str) == selected_run_id]
if selected_run.empty:
    raise RuntimeError(f'Run id not found: {selected_run_id}')


def for_run(df: pd.DataFrame) -> pd.DataFrame:
    if 'run_id' not in df.columns:
        return df.copy()
    return df[df['run_id'].astype(str) == selected_run_id].copy()


prior_run = for_run(prior_df)
refinement_run = for_run(refinement_df)
query_policy_run = for_run(query_policy_df)
uncertainty_run = for_run(uncertainty_df)
undefined_run = for_run(undefined_df)
ablation_run = for_run(ablation_df)
tasks_run = for_run(tasks_df)

print(f'Selected run_id: {selected_run_id}')
print(f'Datasets: {sorted(tasks_run["dataset"].dropna().unique().tolist())}')
print(f'Tasks: {len(tasks_run)}')
print(f'Methods in refinement: {sorted(refinement_run["method"].dropna().unique().tolist())}')


### Summarize The Selected Run
This cell shows the selected run metadata and the task distribution across datasets and concept types.


In [ ]:
display(selected_run)

task_summary = (
    tasks_run
    .groupby(['dataset', 'concept_kind'], dropna=False)
    .agg(
        tasks=('task_id', 'nunique'),
        mean_images=('image_count', 'mean'),
        total_undefined=('undefined_count', 'sum'),
    )
    .reset_index()
    .sort_values(['dataset', 'concept_kind'])
)
display(task_summary)

fig = px.bar(
    task_summary,
    x='dataset',
    y='tasks',
    color='concept_kind',
    barmode='group',
    title='Task count by dataset and concept type',
)
fig.update_layout(height=420)
fig.show()


### Visualize Prior Axis Quality
This cell aggregates the prior-only results and plots method performance before any user feedback.


In [ ]:
prior_ok = prior_run[prior_run['status'].astype(str) == 'ok'].copy()
prior_summary = (
    prior_ok
    .groupby('method', dropna=False)[['spearman', 'pairwise_acc', 'topk_extreme_precision']]
    .mean()
    .sort_values('spearman', ascending=False)
)
display(prior_summary)

prior_long = (
    prior_ok
    .groupby(['dataset', 'method'], dropna=False)[['spearman', 'pairwise_acc', 'topk_extreme_precision']]
    .mean()
    .reset_index()
)
display(prior_long)

fig = px.bar(
    prior_summary.reset_index(),
    x='method',
    y='spearman',
    color='method',
    title='Mean prior Spearman by method',
)
fig.update_layout(showlegend=False, height=430)
fig.show()

fig = px.bar(
    prior_long,
    x='dataset',
    y='spearman',
    color='method',
    barmode='group',
    title='Prior Spearman by dataset',
)
fig.update_layout(height=430)
fig.show()


### Plot Sparse-Feedback Learning Curves
This cell plots the main refinement curves as a function of the interaction budget.


In [ ]:
main_steps = refinement_run[
    (refinement_run['row_type'].astype(str) == 'step')
    & (refinement_run['variant'].astype(str) == 'main')
].copy()

main_steps['metric'] = main_steps['spearman_defined'].where(
    main_steps['spearman_defined'].notna(),
    main_steps['spearman_all'],
)

curve_mean = (
    main_steps
    .groupby(['method', 'interaction_count'], dropna=False)['metric']
    .mean()
    .reset_index()
    .sort_values(['method', 'interaction_count'])
)
display(curve_mean)

fig = px.line(
    curve_mean,
    x='interaction_count',
    y='metric',
    color='method',
    markers=True,
    title='Main refinement learning curves',
)
fig.update_layout(height=460, yaxis_title='Mean Spearman')
fig.show()

curve_dataset = (
    main_steps
    .groupby(['dataset', 'method', 'interaction_count'], dropna=False)['metric']
    .mean()
    .reset_index()
    .sort_values(['dataset', 'method', 'interaction_count'])
)

fig = px.line(
    curve_dataset,
    x='interaction_count',
    y='metric',
    color='method',
    facet_col='dataset',
    facet_col_wrap=2,
    markers=True,
    title='Main refinement learning curves by dataset',
)
fig.update_layout(height=560, yaxis_title='Mean Spearman')
fig.show()


### Compare Query Policies
This cell compares random, diversity, uncertainty, and hybrid selection using both AULC summaries and final-budget performance.


In [ ]:
query_summary = query_policy_run[query_policy_run['row_type'].astype(str) == 'summary'].copy()
query_summary = query_summary.rename(columns={'spearman_all': 'aulc_spearman'})
query_summary_mean = (
    query_summary
    .groupby(['method', 'policy'], dropna=False)['aulc_spearman']
    .mean()
    .reset_index()
)
display(query_summary_mean.sort_values(['method', 'aulc_spearman'], ascending=[True, False]))

query_steps = query_policy_run[query_policy_run['row_type'].astype(str) == 'step'].copy()
max_budget = int(query_steps['interaction_count'].max()) if not query_steps.empty else 0
query_final = query_steps[query_steps['interaction_count'] == max_budget].copy()
query_final['metric'] = query_final['spearman_defined'].where(
    query_final['spearman_defined'].notna(),
    query_final['spearman_all'],
)
query_final_mean = (
    query_final
    .groupby(['method', 'policy'], dropna=False)['metric']
    .mean()
    .reset_index()
)
display(query_final_mean.sort_values(['method', 'metric'], ascending=[True, False]))

fig = px.imshow(
    query_summary_mean.pivot(index='method', columns='policy', values='aulc_spearman'),
    aspect='auto',
    color_continuous_scale='Blues',
    title='Query policy comparison by AULC',
)
fig.update_layout(height=460)
fig.show()

fig = px.imshow(
    query_final_mean.pivot(index='method', columns='policy', values='metric'),
    aspect='auto',
    color_continuous_scale='Blues',
    title=f'Query policy comparison at final budget ({max_budget})',
)
fig.update_layout(height=460)
fig.show()


### Inspect Uncertainty Diagnostics
This cell visualizes how uncertainty tracks error and how performance changes with coverage-risk filtering.


In [ ]:
uncert_error = uncertainty_run[uncertainty_run['row_type'].astype(str) == 'error_prediction'].copy()
uncert_cov = uncertainty_run[uncertainty_run['row_type'].astype(str) == 'coverage_risk'].copy()

if not uncert_error.empty:
    uncert_error_mean = (
        uncert_error
        .groupby(['method', 'interaction_count'], dropna=False)[['corr_uncert_abs_error', 'auroc_future_correction']]
        .mean()
        .reset_index()
    )
    display(uncert_error_mean)

    fig = px.line(
        uncert_error_mean,
        x='interaction_count',
        y='corr_uncert_abs_error',
        color='method',
        markers=True,
        title='Uncertainty vs absolute error correlation',
    )
    fig.update_layout(height=430)
    fig.show()

    fig = px.line(
        uncert_error_mean,
        x='interaction_count',
        y='auroc_future_correction',
        color='method',
        markers=True,
        title='Uncertainty AUROC for future correction',
    )
    fig.update_layout(height=430)
    fig.show()

if not uncert_cov.empty:
    cov_latest_step = int(uncert_cov['interaction_count'].max())
    uncert_cov_latest = uncert_cov[uncert_cov['interaction_count'] == cov_latest_step].copy()
    fig = px.line(
        uncert_cov_latest,
        x='coverage',
        y='spearman',
        color='method',
        markers=True,
        title=f'Coverage-risk curves at interaction {cov_latest_step}',
    )
    fig.update_layout(height=430)
    fig.show()


### Compare Undefined Supervision
This cell compares with-undefined and without-undefined runs on tasks that expose ambiguous middle regions.


In [ ]:
undef_summary = undefined_run[undefined_run['row_type'].astype(str) == 'summary'].copy()
undef_summary = undef_summary.rename(columns={'spearman_all': 'aulc_spearman'})
undef_group = (
    undef_summary
    .groupby(['method', 'variant'], dropna=False)['aulc_spearman']
    .mean()
    .reset_index()
)
display(undef_group.sort_values(['method', 'variant']))

undef_steps = undefined_run[undefined_run['row_type'].astype(str) == 'step'].copy()
if not undef_steps.empty:
    undef_steps['metric'] = undef_steps['spearman_defined'].where(
        undef_steps['spearman_defined'].notna(),
        undef_steps['spearman_all'],
    )
    fig = px.line(
        undef_steps.groupby(['variant', 'method', 'interaction_count'], dropna=False)['metric'].mean().reset_index(),
        x='interaction_count',
        y='metric',
        color='method',
        line_dash='variant',
        markers=True,
        title='Undefined supervision comparison',
    )
    fig.update_layout(height=460, yaxis_title='Mean Spearman')
    fig.show()

fig = px.bar(
    undef_group,
    x='method',
    y='aulc_spearman',
    color='variant',
    barmode='group',
    title='Undefined supervision AULC comparison',
)
fig.update_layout(height=430)
fig.show()


### Summarize Ablations
This cell summarizes the ablation runs so the contribution of each modeling variant is easy to compare.


In [ ]:
abl_summary = ablation_run[ablation_run['row_type'].astype(str) == 'summary'].copy()
abl_summary = abl_summary.rename(columns={'spearman_all': 'aulc_spearman'})
abl_mean = (
    abl_summary
    .groupby('method', dropna=False)['aulc_spearman']
    .mean()
    .reset_index()
    .sort_values('aulc_spearman', ascending=False)
)
display(abl_mean)

abl_dataset = (
    abl_summary
    .groupby(['dataset', 'method'], dropna=False)['aulc_spearman']
    .mean()
    .reset_index()
)
display(abl_dataset)

fig = px.bar(
    abl_mean,
    x='method',
    y='aulc_spearman',
    color='method',
    title='Ablation AULC summary',
)
fig.update_layout(showlegend=False, height=430)
fig.show()

fig = px.bar(
    abl_dataset,
    x='dataset',
    y='aulc_spearman',
    color='method',
    barmode='group',
    title='Ablation AULC by dataset',
)
fig.update_layout(height=430)
fig.show()
